# 01 - Environment Setup and Model Loading

Covers the environment and model-loading stage, and ends with a smoke test to confirm the inference path works.

**Colab GPU:** prefer L4 or A100.

**Colab runtime version:** pin to 2026.07.


## 1. Dependencies

The interpreter version is asserted before the install because it decides whether
the pinned set can be installed at all. `tokenizers==0.19.1`, which OpenVLA's
remote modelling code requires, ships no wheel beyond CPython 3.12 and cannot be
built from source in the runtime, so on a Python 3.13 runtime the install fails and
the session silently keeps the pre-installed transformers, whose 5.x releases no
longer expose `AutoModelForVision2Seq`. 

Key install properties:

- `--only-binary=:all:` forbids a source build, so a missing wheel stops the
  install instead of failing late inside a compiler and leaving the pre-installed
  versions in place.
- torch is never reinstalled. The Colab build is matched to its CUDA driver, and
  overriding it tends to break GPU support.
- protobuf is bounded above as well as below (`>=6.31.1,<7`), mirroring
  `data.PROTOBUF_SPEC`. An unbounded specifier resolves to protobuf 7, which
  removed the descriptor API that older generated modules call.


In [ ]:
import subprocess
import sys

REQUIRED_PYTHON = (3, 12)          # highest version with wheels for the pinned set
COLAB_RUNTIME_VERSION = '2026.07'  # last runtime version shipping that interpreter

assert sys.version_info[:2] == REQUIRED_PYTHON, (
    f'Python {sys.version_info.major}.{sys.version_info.minor} is active, but the pinned '
    f'dependencies require Python {REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]}. Set Runtime > '
    f'Change runtime type > Runtime version to {COLAB_RUNTIME_VERSION}, then reconnect and '
    f'run this notebook from the top.'
)

PINS = [
    'transformers==4.40.1',
    'tokenizers==0.19.1',
    'timm==0.9.10',
    'huggingface_hub==0.23.4',
    'accelerate==0.30.1',
    'bitsandbytes>=0.45.0',
    'protobuf>=6.31.1,<7',
]
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--only-binary=:all:', *PINS],
    check=True,
)
print(f'Python {sys.version.split()[0]}')
print('installed: ' + ' '.join(PINS))

Python 3.12.13
installed: transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 huggingface_hub==0.23.4 accelerate==0.30.1 bitsandbytes>=0.45.0 protobuf>=6.31.1,<7


## 2. Verify the installed set

OpenVLA's remote modelling code raises on an unsupported timm and only warns about
a transformers or tokenizers mismatch, which would otherwise be lost in the load
output. Both the version on disk and the version of any module already imported are
checked, because the two can disagree. An import that predates the install keeps the
old code, and the metadata read here would then describe files the session is not
using. That is the one condition requiring a restart, so it is named explicitly
rather than left to surface as an error inside the load.


In [ ]:
import re
import sys
from importlib.metadata import version

EXPECTED = {
    'transformers': '4.40.1',
    'tokenizers': '0.19.1',
    'timm': '0.9.10',
    'huggingface_hub': '0.23.4',
    'accelerate': '0.30.1',
}
# Mirrors data.PROTOBUF_MIN and data.PROTOBUF_MAX_EXCLUSIVE
PROTOBUF_MIN, PROTOBUF_MAX_EXCLUSIVE = (6, 31, 1), (7, 0, 0)

installed = {name: version(name) for name in EXPECTED}
for name, found in installed.items():
    print(f'{name:16s} {found}')

protobuf_module = sys.modules.get('google.protobuf')
protobuf_in_use = getattr(protobuf_module, '__version__', '') or version('protobuf')
print(f'{"protobuf":16s} {protobuf_in_use}')

mismatched = {n: v for n, v in installed.items() if v != EXPECTED[n]}
assert not mismatched, (
    'Installed versions differ from the pin: '
    + ', '.join(f'{n} {v} (expected {EXPECTED[n]})' for n, v in mismatched.items())
    + '. Re-run the install cell and check it reported no error.'
)

protobuf_parts = tuple(int(part) for part in re.findall(r'\d+', protobuf_in_use)[:3])
assert PROTOBUF_MIN <= protobuf_parts < PROTOBUF_MAX_EXCLUSIVE, (
    f'The protobuf runtime in use ({protobuf_in_use}) is outside the window the '
    f'TensorFlow Datasets stack supports. A runtime inside the window is now '
    f'installed, so restart the session (Runtime > Restart session) and run from '
    f'the top.'
)

stale = {}
for name in EXPECTED:
    module = sys.modules.get(name)
    live = getattr(module, '__version__', '') if module is not None else ''
    if live and live != EXPECTED[name]:
        stale[name] = live
assert not stale, (
    'These packages were imported before the install and the session is still '
    'running the earlier code: '
    + ', '.join(f'{n} {v} (expected {EXPECTED[n]})' for n, v in stale.items())
    + '. Restart the session (Runtime > Restart session) and run from the top.'
)

transformers     4.40.1
tokenizers       0.19.1
timm             0.9.10
huggingface_hub  0.23.4
accelerate       0.30.1
protobuf         6.33.6


## 3. Check the GPU

Compute capability 8.0 or above is required for native bf16 and FlashAttention-2.
Precision is fixed rather than derived from the hardware, since OpenVLA takes the
argmax over discretised action bins and a dtype change between runs can flip a
near-boundary bin, so the loader refuses a bf16 run on an older GPU rather than
falling back.


In [3]:
import torch

assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > GPU.'
p = torch.cuda.get_device_properties(0)
print(f'torch {torch.__version__}')
print(f'GPU: {p.name}  sm_{p.major}{p.minor}  {p.total_memory/1024**3:.1f} GB')
print('FlashAttention-2 / native bf16 available:', (p.major, p.minor) >= (8, 0))

torch 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB  sm_80  39.5 GB
FlashAttention-2 / native bf16 available: True


## 4. Mount Google Drive

Drive holds the Hugging Face cache, so the 7B weights are downloaded once and
reused across sessions. `HF_HOME` has to be set before anything imports the hub,
which is why it is bound here rather than alongside the model load.


In [4]:
import os

from google.colab import drive

drive.mount('/content/drive')

os.environ['HF_HOME'] = '/content/drive/MyDrive/openvla_cache/hf'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
SMOKE_CSV = '/content/smoke_predictions.csv'

print('HF cache  ->', os.environ['HF_HOME'])
print('smoke log ->', SMOKE_CSV)

Mounted at /content/drive
HF cache  -> /content/drive/MyDrive/openvla_cache/hf
smoke log -> /content/smoke_predictions.csv


## 5. Import the project code

Clones the project code from GitHub into the runtime and imports the loader,
inference, and logging functions from there, so the code always matches the
pushed commit.

The import cache is cleared for every project module, not only the one imported
here. 

In [5]:
import sys, os, importlib, subprocess

REPO_URL = 'https://github.com/LewisTL/ECS8056.git'
BRANCH = 'master'
REPO_DIR = '/content/ECS8056'

def sync_repo():
    """Clone or hard-refresh the repository so it matches origin/BRANCH."""
    token = os.environ.get('GITHUB_TOKEN', '')
    url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', url],
                       check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--depth', '1',
                        'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', '--quiet',
                        f'origin/{BRANCH}'], check=True)
    else:
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch',
                        BRANCH, url, REPO_DIR], check=True)
    return subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
                          capture_output=True, text=True).stdout.strip()


commit = sync_repo()
module_dir = REPO_DIR
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
for _m in ('action_bins', 'prediction_log', 'model', 'data', 'controls',
           'compose_scenes', 'export_pairs', 'detect_duplicates', 'analysis'):
    sys.modules.pop(_m, None)
importlib.invalidate_caches()

from model import (append_prediction_log, describe_action_space, load_openvla,
                   predict_action, run_metadata)
print(f'imported model.py from {module_dir} @ {commit}')

imported model.py from /content/ECS8056 @ a449e9b


## 6. Load OpenVLA-7B

Loaded in 4-bit NF4 with bf16 compute. The first run downloads the checkpoint into
the Drive cache and takes several minutes. Later sessions read it back.

In [6]:
processor, vla, compute_dtype = load_openvla(quantize_4bit=True, precision='bf16')
meta = run_metadata(compute_dtype)   # GPU, dtype, seed, library versions for logging
print(meta)

[load_openvla] GPU: NVIDIA A100-SXM4-40GB (sm_80, 39.5 GB) | precision=bf16 | attn=eager | 4bit=True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:99: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[load_openvla] Loaded. GPU memory allocated: 4.08 GB
{'gpu_name': 'NVIDIA A100-SXM4-40GB', 'gpu_capability': 'sm_80', 'dtype': 'bfloat16', 'seed': 42, 'torch': '2.11.0+cu128', 'transformers': '4.40.1', 'bitsandbytes': '0.50.1'}


## 7. Action space and readout constants

The continuous readout used by the probe depends on constants that live in
OpenVLA's remote modelling code and can move between revisions. They are read from the loaded model rather than assumed, and printed
here so a revision change is visible at the point the model is first loaded.

`bin_width` is the resolution floor of the argmax readout on each dimension. Two
predictions closer together than this cannot differ in the executable action, no
matter how differently the model treats them.


In [7]:
space = describe_action_space(vla)
for key, value in space.items():
    if isinstance(value, list):
        print(f'{key:20} ' + ' '.join(f'{v:+.5f}' if isinstance(v, float) else str(v)
                                      for v in value))
    else:
        print(f'{key:20} {value}')
print()
print(f"lateral (dx) bin width: {space['bin_width'][0]:.6f}")

unnorm_key           bridge_orig
action_dim           7
n_bins               255
vocab_size           32000
bin_center_first     -0.996078431372549
bin_center_last      0.996078431372549
action_token_id_min  31745
action_token_id_max  31999
q01                  -0.02873 -0.04170 -0.02609 -0.08092 -0.09289 -0.20718 +0.00000
q99                  +0.02831 +0.04086 +0.04016 +0.08192 +0.07793 +0.20383 +1.00000
mask                 True True True True True True False
bin_width            +0.00022 +0.00033 +0.00026 +0.00064 +0.00067 +0.00162 +0.00394

lateral (dx) bin width: 0.000225


## 8. Smoke test

A synthetic frame is used here. The only checks are that the pipeline
produces a well-formed 7-DoF action `[dx, dy, dz, droll, dpitch, dyaw, gripper]`
and that the row reaches the log. 

In [8]:
import numpy as np
from PIL import Image

dummy = Image.fromarray(
    (np.random.default_rng(0).random((224, 224, 3)) * 255).astype(np.uint8))

instruction = 'pick up the object on the left'
action = predict_action(processor, vla, dummy, instruction, compute_dtype)
append_prediction_log(SMOKE_CSV, action, instruction, meta,
                      scene_id='smoke', spatial_term='left')

assert action.shape == (7,), f'Expected a 7-DoF vector, got {action.shape}'
print('Action shape :', action.shape)
print('Action vector:', np.round(action, 4))
print('Logged to    :', SMOKE_CSV)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


Action shape : (7,)
Action vector: [-0.0029  0.0161 -0.006  -0.0014  0.0086  0.203   0.    ]
Logged to    : /content/smoke_predictions.csv
